In [1]:
import sqlite3                                 
import pandas as pd                             
con = sqlite3.connect(r'C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\data\raw\airtel.db') 
 
def q(sql):                                     
    return pd.read_sql(sql, con)  

In [2]:
def run_sql_file(path):
    with open(path,'r') as f:
        script=f.read()
    con.executescript(script)
    con.commit()
    print('ran',path)

In [3]:

q("""SELECT name FROM sqlite_master                 
WHERE  type = 'table'                  
ORDER  BY name; """)

,name
0,circles
1,complaints
2,customers
3,plans
4,recharges
5,usage_daily


In [4]:
q("""SELECT sql
FROM sqlite_master
WHERE name = 'circles'; """)

,sql
0,"CREATE TABLE ""circles"" (\n""circle_id"" INTEGER,..."


In [5]:
q("""SELECT COUNT(DISTINCT region) AS distinct_,      -- Distinct Count
       SUM(region IS NULL)    AS nulls_,         -- Step 1A tool 1
       SUM(TRIM(region) = '') AS blanks_,        -- hidden blanks
       ROUND(100.0 * AVG(region IS NULL          -- % Missing:
             OR TRIM(region) = ''), 2)           -- NULLs and blanks
         AS pct_missing,                         -- together
       MIN(region) AS smallest,                  -- one end
       MAX(region) AS largest                    -- the other end
FROM   circles;  """)

,distinct_,nulls_,blanks_,pct_missing,smallest,largest
0,4,0,0,0.0,East,West


In [6]:
q("""SELECT COUNT(*) AS rows_,
       COUNT(DISTINCT customer_id) AS distinct_
FROM customers;""")

,rows_,distinct_
0,22000,22000


In [7]:
q("""SELECT AVG(is_churned) * 100 AS churn_rate
FROM customers;""")

,churn_rate
0,23.754545


In [8]:
q("""SELECT AVG(c.is_churned) * 100 AS churn_rate
FROM customers c
JOIN complaints co
    ON co.customer_id = c.customer_id;""")

,churn_rate
0,27.547409


In [9]:
q("""SELECT AVG(c.is_churned) * 100 AS churn_rate
FROM customers c
LEFT JOIN complaints co
    ON co.customer_id = c.customer_id;""")

,churn_rate
0,26.0576


In [10]:
q("""SELECT c.customer_id, c.connection_type,         -- who
       (SELECT COUNT(*) FROM recharges r         -- their payments
        WHERE r.customer_id = c.customer_id)     -- one value each
         AS payments                             -- the evidence
FROM   customers c                               -- all 22,000
WHERE  NOT EXISTS (SELECT 1 FROM usage_daily u   -- no usage at all
                   WHERE u.customer_id           -- for this
                       = c.customer_id); """)

,customer_id,connection_type,payments
0,266,Prepaid,1
1,10684,Prepaid,29
2,13508,Prepaid,8
3,19111,Prepaid,39


In [11]:
q("""WITH l AS (SELECT customer_id,                   -- roll up first
             MAX(usage_date) AS last_use         -- their last day
           FROM usage_daily GROUP BY customer_id) -- one each
SELECT c.is_churned, COUNT(*) AS subs,           -- the two groups
       ROUND(AVG(julianday('2025-06-30')         -- days since
             - julianday(l.last_use)), 2)        -- their last use
         AS avg_quiet,                           -- the average
       MIN(julianday('2025-06-30')               -- the quietest
             - julianday(l.last_use)) AS min_,   -- shortest
       MAX(julianday('2025-06-30')               -- and the
             - julianday(l.last_use)) AS max_    -- longest
FROM   customers c JOIN l                        -- 21,996 with usage
       ON l.customer_id = c.customer_id          -- one each
GROUP  BY c.is_churned;  """)

,is_churned,subs,avg_quiet,min_,max_
0,0,16772,2.66,0.0,143.0
1,1,5224,87.80,7.0,180.0


In [15]:
q("""SELECT MIN(data_mb) AS smallest                  -- the floor: 1.9
FROM   usage_daily;                              -- 1.33m rows
            -- none
""")

,smallest
0,1.9


In [14]:
q("""SELECT COUNT(*) AS zero_rows                     -- how many zeros
FROM   usage_daily WHERE data_mb = 0;""")

,zero_rows
0,0
